# 1. Exploratory Data Analysis & Feature Engineering

## Project Overview
This notebook performs comprehensive EDA on the **Kaggle House Prices** dataset and engineers new features for predictive modeling.

**Dataset:** House Prices - Advanced Regression Techniques  
**Task:** Predict house sale prices (Regression)  
**Target Variable:** `SalePrice`

In [ ]:
# Standard imports
import sys
sys.path.insert(0, '..')  # Add parent directory to path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Project modules
from src.data_load import load_data, split_data, get_dataset_config, validate_dataset
from src.preprocessing import (
    engineer_features, handle_missing_values, detect_outliers,
    build_preprocessing_pipeline, get_feature_lists
)
from src.utils import setup_logging, save_figure

# Setup
logger = setup_logging()
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

# Configuration
DATASET_NAME = 'house-prices-advanced-regression-techniques'
DATA_PATH = Path('../data/raw/')
FIGURES_PATH = Path('../figures/')
FIGURES_PATH.mkdir(parents=True, exist_ok=True)

print('Setup complete!')

## 1.1 Data Loading

We load the House Prices dataset and perform initial inspection.

In [ ]:
# Load dataset
config = get_dataset_config(DATASET_NAME)
print(f"Dataset: {DATASET_NAME}")
print(f"Task Type: {config['task_type']}")
print(f"Target Column: {config['target_column']}")

df = load_data(DATA_PATH, DATASET_NAME)
print(f"\nShape: {df.shape}")
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
df.head()

In [ ]:
# Dataset info
print("=" * 60)
print("DATASET INFO")
print("=" * 60)
print(f"\nData Types:")
print(df.dtypes.value_counts())
print(f"\nNumerical columns: {len(df.select_dtypes(include=[np.number]).columns)}")
print(f"Categorical columns: {len(df.select_dtypes(exclude=[np.number]).columns)}")
print(f"\nBasic Statistics:")
df.describe()

## 1.2 Missing Value Analysis

Understanding missing data patterns is crucial for choosing imputation strategies.

In [ ]:
# Missing value analysis
missing_count = df.isnull().sum()
missing_pct = (missing_count / len(df)) * 100

missing_df = pd.DataFrame({
    'Column': df.columns,
    'Missing Count': missing_count.values,
    'Missing %': missing_pct.values,
    'Data Type': df.dtypes.values
})

missing_df = missing_df[missing_df['Missing Count'] > 0]
missing_df = missing_df.sort_values('Missing %', ascending=False)

print(f"Columns with missing values: {len(missing_df)} out of {len(df.columns)}")
print(f"Total missing values: {df.isnull().sum().sum()}")
missing_df

In [ ]:
# Visualize missing values
if len(missing_df) > 0:
    fig, ax = plt.subplots(figsize=(12, 6))
    bars = ax.barh(missing_df['Column'], missing_df['Missing %'], color='coral')
    ax.set_xlabel('Missing Percentage (%)', fontsize=12)
    ax.set_title('Missing Values by Feature', fontsize=14, fontweight='bold')
    ax.invert_yaxis()
    for bar, pct in zip(bars, missing_df['Missing %']):
        ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                f'{pct:.1f}%', va='center', fontsize=9)
    plt.tight_layout()
    save_figure(fig, 'missing_values', FIGURES_PATH)
    plt.show()
else:
    print('No missing values found!')

## 1.3 Target Variable Analysis

Analyzing the distribution of `SalePrice` to understand our prediction target.

In [ ]:
target_col = config['target_column']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Distribution
axes[0].hist(df[target_col], bins=50, edgecolor='black', color='steelblue', alpha=0.7)
axes[0].set_title(f'{target_col} Distribution', fontsize=13, fontweight='bold')
axes[0].set_xlabel(target_col)
axes[0].set_ylabel('Frequency')

# Log-transformed distribution
axes[1].hist(np.log1p(df[target_col]), bins=50, edgecolor='black', color='seagreen', alpha=0.7)
axes[1].set_title(f'Log({target_col}) Distribution', fontsize=13, fontweight='bold')
axes[1].set_xlabel(f'Log({target_col})')
axes[1].set_ylabel('Frequency')

# Box plot
axes[2].boxplot(df[target_col].dropna(), vert=True)
axes[2].set_title(f'{target_col} Box Plot', fontsize=13, fontweight='bold')
axes[2].set_ylabel(target_col)

plt.tight_layout()
save_figure(fig, 'target_distribution', FIGURES_PATH)
plt.show()

# Summary stats
print(f"\n{target_col} Summary Statistics:")
print(df[target_col].describe())
print(f"\nSkewness: {df[target_col].skew():.4f}")
print(f"Kurtosis: {df[target_col].kurtosis():.4f}")

## 1.4 Numerical Feature Distributions

Plotting histograms for all numerical features to understand their distributions.

In [ ]:
numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()
n_cols = 4
n_rows = (len(numerical_cols) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 3 * n_rows))
axes = axes.flatten()

for idx, col in enumerate(numerical_cols):
    ax = axes[idx]
    df[col].hist(bins=30, ax=ax, edgecolor='black', alpha=0.7)
    ax.set_title(col, fontsize=10, fontweight='bold')
    ax.tick_params(labelsize=8)

for idx in range(len(numerical_cols), len(axes)):
    axes[idx].set_visible(False)

plt.suptitle('Numerical Feature Distributions', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
save_figure(fig, 'numerical_distributions', FIGURES_PATH)
plt.show()

## 1.5 Categorical Feature Distributions

In [ ]:
categorical_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()
print(f"Number of categorical features: {len(categorical_cols)}")

if len(categorical_cols) > 0:
    n_cols_cat = 3
    n_rows_cat = (len(categorical_cols) + n_cols_cat - 1) // n_cols_cat
    fig, axes = plt.subplots(n_rows_cat, n_cols_cat, figsize=(18, 4 * n_rows_cat))
    axes = axes.flatten()

    for idx, col in enumerate(categorical_cols):
        ax = axes[idx]
        value_counts = df[col].value_counts().head(10)
        value_counts.plot(kind='bar', ax=ax, color='steelblue', alpha=0.7)
        ax.set_title(col, fontsize=10, fontweight='bold')
        ax.tick_params(labelsize=7, rotation=45)

    for idx in range(len(categorical_cols), len(axes)):
        axes[idx].set_visible(False)

    plt.suptitle('Categorical Feature Distributions (Top 10 values)', fontsize=16, fontweight='bold', y=1.01)
    plt.tight_layout()
    save_figure(fig, 'categorical_distributions', FIGURES_PATH)
    plt.show()

## 1.6 Correlation Analysis

Computing and visualizing the correlation matrix to identify feature relationships and multicollinearity.

In [ ]:
# Correlation heatmap
numerical_df = df.select_dtypes(include=[np.number])
corr_matrix = numerical_df.corr()

fig, ax = plt.subplots(figsize=(16, 14))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=False,
    cmap='coolwarm',
    center=0,
    square=True,
    linewidths=0.5,
    ax=ax
)
ax.set_title('Correlation Matrix (Numerical Features)', fontsize=16, fontweight='bold')
plt.tight_layout()
save_figure(fig, 'correlation_matrix', FIGURES_PATH)
plt.show()

In [ ]:
# Top correlations with target
if target_col in corr_matrix.columns:
    target_corr = corr_matrix[target_col].drop(target_col).abs().sort_values(ascending=False)
    
    fig, ax = plt.subplots(figsize=(12, 8))
    top_20 = target_corr.head(20)
    colors = plt.cm.RdYlGn(top_20.values / top_20.values.max())
    ax.barh(range(len(top_20)), top_20.values, color=colors)
    ax.set_yticks(range(len(top_20)))
    ax.set_yticklabels(top_20.index)
    ax.set_xlabel('Absolute Correlation', fontsize=12)
    ax.set_title(f'Top 20 Features Correlated with {target_col}', fontsize=14, fontweight='bold')
    ax.invert_yaxis()
    plt.tight_layout()
    save_figure(fig, 'top_correlations', FIGURES_PATH)
    plt.show()
    
    print("\nTop 10 features correlated with target:")
    print(target_corr.head(10))

## 1.7 Outlier Detection

Using the IQR method to identify outliers in numerical features.

In [ ]:
# Detect outliers using IQR method
outlier_mask = detect_outliers(df, method='iqr', threshold=1.5)
outlier_counts = outlier_mask.sum().sort_values(ascending=False)
outlier_pcts = (outlier_counts / len(df) * 100)

outlier_summary = pd.DataFrame({
    'Outlier Count': outlier_counts,
    'Outlier %': outlier_pcts
})
outlier_summary = outlier_summary[outlier_summary['Outlier Count'] > 0]

print(f"Features with outliers: {len(outlier_summary)}")
outlier_summary.head(15)

In [ ]:
# Visualize outliers in top features
top_outlier_features = outlier_counts[outlier_counts > 0].head(6).index.tolist()

if len(top_outlier_features) > 0:
    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    axes = axes.flatten()
    
    for idx, col in enumerate(top_outlier_features[:6]):
        if idx < len(axes):
            axes[idx].boxplot(df[col].dropna())
            axes[idx].set_title(f'{col}\n({outlier_counts[col]} outliers)', fontsize=10, fontweight='bold')
    
    for idx in range(len(top_outlier_features), len(axes)):
        axes[idx].set_visible(False)
    
    plt.suptitle('Outlier Analysis (Box Plots)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    save_figure(fig, 'outlier_boxplots', FIGURES_PATH)
    plt.show()

## 1.8 Feature Engineering

Creating domain-specific features to improve model performance.

**Rationale for engineered features:**
- **TotalSF**: Combined square footage (basement + 1st floor + 2nd floor) is a strong predictor
- **TotalBath**: Consolidates all bathroom types into a single metric
- **HouseAge**: Age of the house affects condition and buyer preferences
- **IsNew**: Binary flag for recently built houses (premium pricing)
- **QualityArea**: Interaction between overall quality and living area

In [ ]:
# Apply feature engineering
print(f"Original features: {df.shape[1]}")
df_engineered = engineer_features(df, dataset_type='house-prices')
print(f"After engineering: {df_engineered.shape[1]}")

# Show new features
new_features = [c for c in df_engineered.columns if c not in df.columns]
print(f"\nNew features created: {new_features}")

if new_features:
    print("\nNew feature statistics:")
    print(df_engineered[new_features].describe())

In [ ]:
# Visualize engineered features vs target
eng_features_to_plot = [f for f in new_features if f in df_engineered.select_dtypes(include=[np.number]).columns]

if len(eng_features_to_plot) > 0:
    n = min(len(eng_features_to_plot), 6)
    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    axes = axes.flatten()

    for idx, feat in enumerate(eng_features_to_plot[:n]):
        axes[idx].scatter(df_engineered[feat], df_engineered[target_col], alpha=0.3, s=10)
        axes[idx].set_xlabel(feat, fontsize=10)
        axes[idx].set_ylabel(target_col, fontsize=10)
        axes[idx].set_title(f'{feat} vs {target_col}', fontsize=11, fontweight='bold')

    for idx in range(n, len(axes)):
        axes[idx].set_visible(False)

    plt.suptitle('Engineered Features vs Target', fontsize=14, fontweight='bold')
    plt.tight_layout()
    save_figure(fig, 'engineered_features_scatter', FIGURES_PATH)
    plt.show()

## 1.9 Preprocessing Pipeline

Building a scikit-learn preprocessing pipeline that:
1. Imputes missing values (median for numerical, most frequent for categorical)
2. Scales numerical features (StandardScaler)
3. Encodes categorical features (OneHotEncoder)

**Important:** The pipeline is fit only on training data to prevent data leakage.

In [ ]:
# Handle missing values first
df_clean = handle_missing_values(df_engineered)
print(f"Missing values after imputation: {df_clean.isnull().sum().sum()}")

# Split data BEFORE fitting the pipeline (prevents data leakage)
X_train, X_val, X_test, y_train, y_val, y_test = split_data(
    df_clean,
    target_column=target_col,
    test_size=0.2,
    val_size=0.1,
    random_state=42,
    stratify=False  # Regression task
)

print(f"\nTrain: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")
print(f"y_train: {y_train.shape}, y_val: {y_val.shape}, y_test: {y_test.shape}")

In [ ]:
# Identify feature types
# Drop ID-like columns
drop_cols = ['Id'] if 'Id' in X_train.columns else []
if drop_cols:
    X_train = X_train.drop(columns=drop_cols, errors='ignore')
    X_val = X_val.drop(columns=drop_cols, errors='ignore')
    X_test = X_test.drop(columns=drop_cols, errors='ignore')

numerical_features = X_train.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=[np.number]).columns.tolist()

print(f"Numerical features: {len(numerical_features)}")
print(f"Categorical features: {len(categorical_features)}")

# Build preprocessing pipeline
preprocessor = build_preprocessing_pipeline(
    numerical_features=numerical_features,
    categorical_features=categorical_features,
    numerical_strategy='standard',
    categorical_strategy='onehot'
)

# Fit on training data ONLY
X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

print(f"\nProcessed shapes:")
print(f"X_train: {X_train_processed.shape}")
print(f"X_val: {X_val_processed.shape}")
print(f"X_test: {X_test_processed.shape}")

In [ ]:
# Save preprocessed data and pipeline for the modeling notebook
import joblib

processed_path = Path('../data/processed/')
processed_path.mkdir(parents=True, exist_ok=True)

# Save pipeline
joblib.dump(preprocessor, processed_path / 'preprocessor.joblib')

# Save processed data
np.save(processed_path / 'X_train.npy', X_train_processed)
np.save(processed_path / 'X_val.npy', X_val_processed)
np.save(processed_path / 'X_test.npy', X_test_processed)
y_train.to_csv(processed_path / 'y_train.csv', index=False)
y_val.to_csv(processed_path / 'y_val.csv', index=False)
y_test.to_csv(processed_path / 'y_test.csv', index=False)

# Save feature names
try:
    feature_names = preprocessor.get_feature_names_out().tolist()
except:
    feature_names = numerical_features + categorical_features

joblib.dump(feature_names, processed_path / 'feature_names.joblib')

print('Saved preprocessed data and pipeline to data/processed/')
print(f'Feature names count: {len(feature_names)}')

## 1.10 Summary & Key Findings

### Key Findings:
1. **Dataset Size:** ~1,460 rows × 81 features (including target)
2. **Missing Values:** Several features have significant missing data (PoolQC, MiscFeature, Alley have >80%)
3. **Target Distribution:** SalePrice is right-skewed, log transformation makes it more normal
4. **Top Correlations:** OverallQual, GrLivArea, GarageCars, GarageArea are most correlated with SalePrice
5. **Feature Engineering:** Created TotalSF, TotalBath, HouseAge, IsNew, QualityArea features

### Next Steps:
- Train supervised models (Baseline, Linear, Random Forest, XGBoost, LightGBM, SVM)
- Perform unsupervised analysis (KMeans, DBSCAN, PCA, t-SNE)
- Apply model interpretability (SHAP, Partial Dependence Plots)